# DeepTrace – Feature Engineering

## Notebook 02

This notebook transforms raw synthetic cybersecurity event logs into high-quality machine learning features for threat detection.

The engineered features created here will be used by:

- Isolation Forest (Baseline Anomaly Detection)
- Transformer Encoder (Behavior Sequence Learning)
- XGBoost (Threat Classification)

By the end of this notebook, we will have a clean, model-ready feature matrix for downstream training and evaluation.

In [36]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

In [37]:
import pandas as pd

df = pd.read_csv("/content/synthetic_cybersecurity_logs.csv")

print(df.shape)
df.head()

(239480, 21)


,event_id,session_id,timestamp,employee_id,employee_name,department,role,office_location,device,network_type,...,event_type,application,resource,risk_baseline,access_level,label,attack_label,is_attack,country,city
0,9a390f5c-93eb-408d-bf79-3454c09ccb8e,6228be5a-83c5-4019-8954-190ae9dc1188,2026-07-24 11:56:30,EMP0045,Alyssa Vance,Executive,Vice President,Mumbai,MacBook,Corporate LAN,...,Email,Outlook,Corporate Mailbox,Low,Standard,Normal,NaN,0,India,Bhopal
1,46511ce4-b6fa-4952-80eb-c5b4c4eaf2bc,f0412e24-0601-4faf-b8ac-1e2edcc9b417,2026-07-08 17:39:54,EMP0212,Mr. William Ward,Finance,Finance Manager,Hyderabad,Windows Laptop,Corporate LAN,...,Email,Outlook,Corporate Mailbox,Medium,Elevated,Normal,NaN,0,India,Bhopal
2,35b78fa2-9290-4488-bf79-f4605cea6c0e,d3888bd5-a2d8-4b31-bcfc-43fb5cc8209f,2026-07-21 10:01:54,EMP0202,Matthew White,Sales,Sales Manager,Bangalore,Mobile,Home WiFi,...,Email,Outlook,Corporate Mailbox,Medium,Elevated,Normal,NaN,0,India,Bhopal
3,cb9dfdd6-255a-4c53-b27c-a48a53a60b12,c35dd141-0ba8-46bf-8575-3702f88527fd,2026-07-20 18:27:07,EMP0137,Scott Greene,Engineering,Senior Engineer,Mumbai,Windows Laptop,Corporate LAN,...,Upload,OneDrive,Cloud Storage,Low,Standard,Normal,NaN,0,India,Bhopal
4,993073c2-bf31-4558-b04a-d64687c7056d,82bfe1cc-ce26-4ba3-9f26-36e3374cdefd,2026-07-06 17:56:17,EMP0313,Robert Burns,Sales,Sales Executive,Pune,Mobile,Home WiFi,...,Email,Outlook,Corporate Mailbox,Low,Standard,Normal,NaN,0,India,Bhopal


In [38]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [39]:
df = df.sort_values(
    ["employee_id", "timestamp"]
).reset_index(drop=True)

## Temporal Feature Engineering

Extract time-based features from the event timestamp. These features help capture user activity patterns and will be useful for anomaly detection and threat classification.

In [40]:
df["hour"] = df["timestamp"].dt.hour
df["minute"] = df["timestamp"].dt.minute
df["second"] = df["timestamp"].dt.second

df["day"] = df["timestamp"].dt.day
df["weekday"] = df["timestamp"].dt.weekday
df["month"] = df["timestamp"].dt.month
df["quarter"] = df["timestamp"].dt.quarter

df["is_weekend"] = (df["weekday"] >= 5).astype(int)

df["business_hours"] = (
    (df["hour"] >= 9) &
    (df["hour"] <= 18)
).astype(int)

In [41]:
df[
    [
        "timestamp",
        "hour",
        "weekday",
        "is_weekend",
        "business_hours",
    ]
].head()

,timestamp,hour,weekday,is_weekend,business_hours
0,2026-06-25 07:18:12,7,3,0,0
1,2026-06-25 08:02:05,8,3,0,0
2,2026-06-25 08:46:35,8,3,0,0
3,2026-06-25 09:36:44,9,3,0,1
4,2026-06-25 10:18:36,10,3,0,1


In [42]:
df["time_since_last_event"] = (
    df.groupby("employee_id")["timestamp"]
      .diff()
      .dt.total_seconds()
)

In [43]:
df[
    [
        "employee_id",
        "timestamp",
        "time_since_last_event"
    ]
].head(10)

,employee_id,timestamp,time_since_last_event
0,EMP0001,2026-06-25 07:18:12,NaN
1,EMP0001,2026-06-25 08:02:05,2633.0
2,EMP0001,2026-06-25 08:46:35,2670.0
3,EMP0001,2026-06-25 09:36:44,3009.0
4,EMP0001,2026-06-25 10:18:36,2512.0
5,EMP0001,2026-06-25 11:01:52,2596.0
6,EMP0001,2026-06-25 11:50:43,2931.0
7,EMP0001,2026-06-25 12:28:56,2293.0
8,EMP0001,2026-06-25 13:08:42,2386.0
9,EMP0001,2026-06-25 13:45:54,2232.0


In [44]:
df["event_number"] = (
    df.groupby("session_id")
      .cumcount() + 1
)

In [45]:
df[
    [
        "session_id",
        "event_type",
        "event_number"
    ]
].head(15)

,session_id,event_type,event_number
0,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Login,1
1,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Admin Action,2
2,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,3
3,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,4
4,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,5
5,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,6
6,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Admin Action,7
7,becefa7e-7ee4-454b-8212-78a9d59e1dc6,VPN Login,8
8,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Database Query,9
9,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,10


In [46]:
session_stats = (
    df.groupby("session_id")["timestamp"]
      .agg(["min", "max", "count"])
      .reset_index()
)

session_stats["session_duration"] = (
    session_stats["max"] - session_stats["min"]
).dt.total_seconds()

session_stats.rename(
    columns={"count": "events_per_session"},
    inplace=True
)

In [47]:
df = df.merge(
    session_stats[
        [
            "session_id",
            "session_duration",
            "events_per_session",
        ]
    ],
    on="session_id",
    how="left",
)

In [48]:
df[
    [
        "session_id",
        "session_duration",
        "events_per_session",
    ]
].head(10)

,session_id,session_duration,events_per_session
0,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
1,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
2,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
3,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
4,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
5,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
6,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
7,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
8,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16
9,becefa7e-7ee4-454b-8212-78a9d59e1dc6,39388.0,16


## Behavioral Feature Engineering

Create behavioral features that summarize user activity within each session. These features capture login patterns, file operations, administrative actions, and other security-relevant behaviors.

In [49]:
event_features = [
    "Login",
    "Login Failed",
    "VPN Login",
    "Download",
    "Upload",
    "Database Query",
    "File Access",
    "Admin Action",
    "PowerShell Execution"
]

for event in event_features:
    column = (
        event.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    df[column] = (
        df["event_type"] == event
    ).astype(int)

In [50]:
df[
    [
        "event_type",
        "login",
        "login_failed",
        "vpn_login",
        "download",
        "upload",
        "database_query",
        "file_access",
        "admin_action",
        "powershell_execution"
    ]
].head(15)

,event_type,login,login_failed,vpn_login,download,upload,database_query,file_access,admin_action,powershell_execution
0,Login,1,0,0,0,0,0,0,0,0
1,Admin Action,0,0,0,0,0,0,0,1,0
2,Application Launch,0,0,0,0,0,0,0,0,0
3,Application Launch,0,0,0,0,0,0,0,0,0
4,Application Launch,0,0,0,0,0,0,0,0,0
5,Application Launch,0,0,0,0,0,0,0,0,0
6,Admin Action,0,0,0,0,0,0,0,1,0
7,VPN Login,0,0,1,0,0,0,0,0,0
8,Database Query,0,0,0,0,0,1,0,0,0
9,Application Launch,0,0,0,0,0,0,0,0,0


In [51]:
session_features = [
    "login",
    "login_failed",
    "vpn_login",
    "download",
    "upload",
    "database_query",
    "file_access",
    "admin_action",
    "powershell_execution"
]

for feature in session_features:
    df[f"{feature}_count"] = (
        df.groupby("session_id")[feature]
          .transform("sum")
    )

In [52]:
df[
    [
        "session_id",
        "event_type",
        "download_count",
        "database_query_count",
        "admin_action_count",
        "powershell_execution_count"
    ]
].head(20)

,session_id,event_type,download_count,database_query_count,admin_action_count,powershell_execution_count
0,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Login,0,3,3,0
1,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Admin Action,0,3,3,0
2,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,0,3,3,0
3,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,0,3,3,0
4,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,0,3,3,0
5,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,0,3,3,0
6,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Admin Action,0,3,3,0
7,becefa7e-7ee4-454b-8212-78a9d59e1dc6,VPN Login,0,3,3,0
8,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Database Query,0,3,3,0
9,becefa7e-7ee4-454b-8212-78a9d59e1dc6,Application Launch,0,3,3,0


In [53]:
df["is_privileged_user"] = (
    df["access_level"] == "Elevated"
).astype(int)

df["is_admin"] = (
    df["role"]
    .str.contains("Manager|Director|Vice President", case=False)
).astype(int)

In [54]:
df[
    [
        "role",
        "access_level",
        "is_privileged_user",
        "is_admin"
    ]
].head(15)

,role,access_level,is_privileged_user,is_admin
0,Incident Responder,Standard,0,0
1,Incident Responder,Standard,0,0
2,Incident Responder,Standard,0,0
3,Incident Responder,Standard,0,0
4,Incident Responder,Standard,0,0
5,Incident Responder,Standard,0,0
6,Incident Responder,Standard,0,0
7,Incident Responder,Standard,0,0
8,Incident Responder,Standard,0,0
9,Incident Responder,Standard,0,0


In [55]:
df["remote_access"] = (
    df["network_type"] != "Corporate LAN"
).astype(int)

In [56]:
df[
    [
        "network_type",
        "remote_access"
    ]
].drop_duplicates()

,network_type,remote_access
0,Corporate LAN,0
7,VPN,1
44,Home WiFi,1


In [57]:
df["high_risk_event"] = df["event_type"].isin([
    "Login Failed",
    "Admin Action",
    "PowerShell Execution",
    "Database Query"
]).astype(int)

In [58]:
df[
    [
        "event_type",
        "high_risk_event"
    ]
].drop_duplicates()

,event_type,high_risk_event
0,Login,0
1,Admin Action,1
2,Application Launch,0
7,VPN Login,0
8,Database Query,1
15,Logout,0
23,File Access,0
513,Print,0
518,Email,0
1080,PowerShell Execution,1


In [59]:
df.describe()

,timestamp,is_attack,hour,minute,second,day,weekday,month,quarter,is_weekend,...,download_count,upload_count,database_query_count,file_access_count,admin_action_count,powershell_execution_count,is_privileged_user,is_admin,remote_access,high_risk_event
count,239480,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,...,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000,239480.000000
mean,2026-07-10 15:45:59.346132992,0.023062,13.739035,29.481886,29.430817,15.498939,2.131122,6.818824,2.818824,0.000004,...,0.603787,0.952768,1.992738,4.895398,1.006272,0.519075,0.307391,0.437226,0.167847,0.150810
min,2026-06-25 06:03:37,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,6.000000,2.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2026-07-02 14:25:04,0.000000,11.000000,14.000000,14.000000,8.000000,1.000000,7.000000,3.000000,0.000000,...,0.000000,0.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2026-07-10 08:09:21,0.000000,14.000000,29.000000,29.000000,16.000000,2.000000,7.000000,3.000000,0.000000,...,0.000000,0.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2026-07-17 14:23:36.750000128,0.000000,16.000000,45.000000,44.000000,23.000000,3.000000,7.000000,3.000000,0.000000,...,0.000000,0.000000,4.000000,7.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000
max,2026-07-24 23:50:34,1.000000,23.000000,59.000000,59.000000,30.000000,6.000000,7.000000,3.000000,1.000000,...,13.000000,13.000000,16.000000,18.000000,13.000000,14.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,0.150102,3.318841,17.319239,17.299932,8.696407,1.424832,0.385165,0.385165,0.002043,...,1.609853,1.950518,2.549466,2.805032,2.121138,1.714870,0.461414,0.496045,0.373731,0.357864


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239480 entries, 0 to 239479
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   event_id                    239480 non-null  object        
 1   session_id                  239480 non-null  object        
 2   timestamp                   239480 non-null  datetime64[ns]
 3   employee_id                 239480 non-null  object        
 4   employee_name               239480 non-null  object        
 5   department                  239480 non-null  object        
 6   role                        239480 non-null  object        
 7   office_location             239480 non-null  object        
 8   device                      239480 non-null  object        
 9   network_type                239480 non-null  object        
 10  authentication              239480 non-null  object        
 11  event_type                  239480 non-

In [61]:
df["time_since_last_event"] = (
    df["time_since_last_event"]
    .fillna(0)
)

In [62]:
df.tail()

,event_id,session_id,timestamp,employee_id,employee_name,department,role,office_location,device,network_type,...,download_count,upload_count,database_query_count,file_access_count,admin_action_count,powershell_execution_count,is_privileged_user,is_admin,remote_access,high_risk_event
239475,31be9f3a-9ac7-4846-982a-4844c850ab90,e1067bb4-6f82-41b7-9676-83515e0ce97b,2026-07-24 17:47:30,EMP0500,Juan Johnson,Executive,Director,Bangalore,MacBook,Corporate LAN,...,0,0,6,3,0,0,0,1,0,1
239476,e3859cbc-d435-4374-96a9-9a2fa13683a8,e1067bb4-6f82-41b7-9676-83515e0ce97b,2026-07-24 18:03:42,EMP0500,Juan Johnson,Executive,Director,Bangalore,MacBook,Corporate LAN,...,0,0,6,3,0,0,0,1,0,0
239477,8df39e3f-d7c7-462d-8cf6-8c3ad0732db8,e1067bb4-6f82-41b7-9676-83515e0ce97b,2026-07-24 18:33:29,EMP0500,Juan Johnson,Executive,Director,Bangalore,MacBook,Corporate LAN,...,0,0,6,3,0,0,0,1,0,1
239478,505284f3-7461-4ede-9461-f5d2f418c036,e1067bb4-6f82-41b7-9676-83515e0ce97b,2026-07-24 18:54:00,EMP0500,Juan Johnson,Executive,Director,Bangalore,MacBook,VPN,...,0,0,6,3,0,0,0,1,1,0
239479,d22eaf38-9b45-4359-a661-36cb72b13589,e1067bb4-6f82-41b7-9676-83515e0ce97b,2026-07-24 19:17:02,EMP0500,Juan Johnson,Executive,Director,Bangalore,MacBook,Corporate LAN,...,0,0,6,3,0,0,0,1,0,0


In [63]:
df.isnull().sum()

,0
event_id,0
session_id,0
timestamp,0
employee_id,0
employee_name,0
department,0
role,0
office_location,0
device,0
network_type,0


## Categorical Feature Encoding

Convert categorical features into numerical values using Label Encoding so they can be used by machine learning models.

In [64]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

categorical_columns = [
    "department",
    "role",
    "office_location",
    "device",
    "network_type",
    "authentication",
    "event_type",
    "application",
    "resource",
    "risk_baseline",
    "access_level"
]

for column in categorical_columns:
    df[column] = label_encoder.fit_transform(df[column])

In [65]:
df[categorical_columns].head()

,department,role,office_location,device,network_type,authentication,event_type,application,resource,risk_baseline,access_level
0,7,9,4,1,0,3,6,1,1,1,2
1,7,9,4,1,0,1,0,0,9,1,2
2,7,9,4,1,0,3,1,29,14,1,2
3,7,9,4,1,0,0,1,7,2,1,2
4,7,9,4,1,0,2,1,12,14,1,2


## Preparing the Final Feature Set

The engineered dataset is prepared for machine learning by separating identifiers, target variables, and input features. This final feature matrix will be used by the anomaly detection and classification models in the subsequent notebooks.

In [66]:
y_binary = df["is_attack"]

y_multiclass = df["attack_label"]

In [67]:
columns_to_drop = [
    "event_id",
    "session_id",
    "timestamp",
    "employee_id",
    "employee_name",
    "label",
    "attack_label",
    "is_attack"
]

X = df.drop(columns=columns_to_drop)

In [68]:
print("Feature Matrix Shape :", X.shape)
print("Binary Labels Shape  :", y_binary.shape)
print("Multiclass Labels Shape :", y_multiclass.shape)

Feature Matrix Shape : (239480, 48)
Binary Labels Shape  : (239480,)
Multiclass Labels Shape : (239480,)


In [74]:
df["foreign_login"] = (df["country"] != "India").astype(int)

In [76]:
columns_to_drop = [
    "event_id",
    "session_id",
    "timestamp",
    "employee_id",
    "employee_name",
    "label",
    "attack_label",
    "is_attack"
]

X = df.drop(columns=columns_to_drop)

In [77]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

for column in ["country", "city"]:
    X[column] = label_encoder.fit_transform(X[column])

In [78]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

In [79]:
print(X_scaled.shape)
print("foreign_login" in X_scaled.columns)

(239480, 49)
True


In [81]:
X_scaled.to_csv("deeptrace_features.csv", index=False)

In [82]:
y_binary.to_csv("deeptrace_binary_labels.csv", index=False)

y_multiclass.to_csv("deeptrace_multiclass_labels.csv", index=False)

In [83]:
import joblib

joblib.dump(scaler, "deeptrace_scaler.pkl")

['deeptrace_scaler.pkl']

In [84]:
processed_df = X_scaled.copy()

processed_df["is_attack"] = y_binary.values
processed_df["attack_label"] = y_multiclass.values

processed_df.to_csv("deeptrace_processed.csv", index=False)